# Fine-tune YOLO for player + ball detection

Thin Colab orchestrator around `scripts/finetune.py`. Designed to run on a Colab GPU runtime.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is sufficient).
2. Add `ROBOFLOW_API_KEY` to Colab Secrets (left sidebar key icon, enable "Notebook access").
3. Fill the `DATASETS` list in **Step 4** with (workspace, project, version) tuples. You can specify multiple datasets—they will be automatically merged into a single training dataset.

Training defaults live in `scripts/finetune.py` (`imgsz=1280`, `batch=4`, 300 epochs, starting from `models/yolo11m.pt`). The script copies `best.pt` to `models/yolo_finetuned.pt`; the final cell triggers a browser download of that file.

## 1. Verify GPU runtime

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## 3. Clone the repo


In [ ]:
!git clone https://github.com/446f6e6e79/player-tracking-in-sports.git
%cd player-tracking-in-sports
#!git checkout motion-detection


## 4. Download and merge datasets from Roboflow

Fill in the `DATASETS` list below with (workspace, project, version) tuples. All datasets will be downloaded and merged into a single training dataset. The API key is read from Colab Secrets.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata
from roboflow import Roboflow
from src.utils.dataset_merge import merge_yolo_datasets

# ============================================================================
# CONFIGURE DATASETS HERE
# ============================================================================
# List of (workspace, project, version) tuples to download and merge.
# Example:
#   DATASETS = [
#       ("workspace1", "project-1", 1),
#       ("workspace1", "project-2", 2),
#       ("workspace2", "project-3", 1),
#   ]
DATASETS = [
    ("<workspace>", "<project>", "<version>"),  # Replace with your dataset
    # ("TODO_WORKSPACE_2", "TODO_PROJECT_2", 1),  # Uncomment and add more datasets
]

In [ ]:
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

# Download all datasets
downloaded_data_yamls = []
for workspace, project, version in DATASETS:
    print(f"Downloading {project} v{version} from {workspace}...")
    dataset = rf.workspace(workspace).project(project).version(version).download("yolo26")

    data_yaml = f"{dataset.location}/data.yaml"
    downloaded_data_yamls.append(data_yaml)
    print(f"Saved to: {data_yaml}")

In [ ]:
# Merge all datasets into one
if len(downloaded_data_yamls) == 1:
    print(f"Single dataset detected, using as-is")
    data_yaml = downloaded_data_yamls[0]
else:
    print(f"Merging {len(downloaded_data_yamls)} datasets...")
    merged_path = merge_yolo_datasets(downloaded_data_yamls, output_dir="/content/merged_dataset")
    data_yaml = str(merged_path)

print(f"Final data.yaml: {data_yaml}")

## 5. Run the finetune script

Defaults from `scripts/finetune.py` apply (50 epochs, `imgsz=1280`, `batch=8`). Override here if needed.

In [ ]:
!python scripts/finetune.py \
    --data "{data_yaml}" \
    --device 0

## 6. Download the fine-tuned weights

Saves `models/yolo_finetuned.pt` to your local machine. Drop it into the repo's `models/` directory to use it from `notebook.ipynb`.

**Note** that, if the **training was interrupted** before completion, the `best.pt` file may not be copied to `models/yolo_finetuned.pt`. In that case, you can download `runs/detect/{name}/``weights/best.pt` and rename it to `yolo_finetuned.pt`.

In [ ]:
from google.colab import files
files.download("models/yolo_finetuned.pt")